# パディングと切り詰めをしてバッチのテンソルを作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/05_subword/padding_truncation.ipynb)

NLP クックブック 05-06 ／ サブワードとトークナイザ

長さの違う複数の文を、モデルに入れられる同じ長さのテンソルにまとめたい。

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
texts = ["短い文。", "これは少し長めの文で、トークン数が多くなります。", "This is an English sentence."]

batch = tokenizer(texts, padding=True, truncation=True, max_length=12, return_tensors="pt")
print(batch["input_ids"])
print(batch["attention_mask"])  # 1 = 本物のトークン、0 = パディング

tensor([[     0,      6,  10119,   2111,   2564,     30,      2,      1,      1,
              1,      1,      1],
        [     0,  85873,  26397,   2846,   8421,    154,   2564,    507,     37,
         232471,   7740,      2],
        [     0,   3293,     83,    142,  14941, 149357,      5,      2,      1,
              1,      1,      1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])


In [2]:
for name, kwargs in {
    "padding=True (バッチ内最長に合わせる)": dict(padding=True),
    "padding='max_length'": dict(padding="max_length", max_length=20),
    "truncation のみ": dict(truncation=True, max_length=8),
}.items():
    print(name, [len(x) for x in tokenizer(texts, **kwargs)["input_ids"]])

padding=True (バッチ内最長に合わせる) [16, 16, 16]
padding='max_length' [20, 20, 20]
truncation のみ [7, 8, 8]


- 学習時は `padding=True` を tokenizer ではなく `DataCollatorWithPadding` に任せると、バッチごとに最短で済んで速い (動的パディング)。
- 生成モデル (GPT 系) は左パディングが必要: `tokenizer.padding_side = "left"`。